In [ ]:
from pathlib import Path
from typing import Literal

import numpy as np
import polars as pl
from ainterviewer.lpm.types import CustomTokens
from rich.console import Console

In [ ]:
def print_interview(
    conversation_id: str,
    messages: pl.DataFrame,
    timestamp_format="%H:%M:%S",
    interviewer: Literal["human", "ai"] = "human",
):
    console = Console(highlight=False)

    interview_transcript = f"""Conversation id: [deep_sky_blue1]{conversation_id}[/deep_sky_blue1]
Interviewer: [deep_sky_blue1]{interviewer}[/deep_sky_blue1]\n\n"""
    for row in messages.filter(conversation_id=conversation_id).iter_rows(named=True):
        role = row["role"]
        role_color = "turquoise4" if role == "ASSISTANT" else "orange_red1"
        content = row["content"]
        if content.strip() in CustomTokens.all:
            interview_transcript += f"\n[purple]{content.strip()}[/purple]\n\n"
        else:
            timestamp = row["created_at"].strftime(timestamp_format)

            prefix = f"[purple]{timestamp}[/purple]"
            if row["section"] is not None:
                interview_position = f"{int(row['section'])} - {int(row['main_question'])} - {int(row['sub_question'])}"
                prefix += f" [yellow]{interview_position}[/yellow]"
            prefix += f" [{role_color}]{role}[/{role_color}]"
            interview_transcript += f"{prefix}:\n{content.strip()}\n\n"

    console.print(interview_transcript, highlight=False)


def calculate_response_times(
    df: pl.DataFrame, who: Literal["interviewer"] | Literal["user"] = "interviewer"
) -> list[pl.Datetime]:
    response_times = []
    for row in df.filter(role="USER").iter_rows(named=True):
        if not (
            next_message := df.filter(
                message_id=row["message_id"] + (-1 if who == "user" else 1)
            )
        ).is_empty():
            assert next_message["role"][0] == "ASSISTANT"
            response_time = (
                next_message["created_at"] - row["created_at"]
                if who == "interviewer"
                else row["created_at"] - next_message["created_at"]
            )
            response_times.append(response_time)

    return response_times

In [ ]:
conversation_dumps = list(sorted(Path("../../data/dumps/").glob("conversation*.csv")))
message_dumps = list(sorted(Path("../../data/dumps/").glob("message*.csv")))

In [ ]:
conversations = pl.read_csv(conversation_dumps[-1])
messages = pl.read_csv(message_dumps[-1])

In [ ]:
db_uri = "sqlite://../../data/dumps/ainterviewer.sqlite"

In [ ]:
messages_query = """SELECT *
	FROM message
	WHERE created_at > '2025-04-30 11:00:00'
"""

conversations = pl.read_database_uri(
    uri=db_uri, query="SELECT * FROM conversation", engine="adbc"
)
messages = pl.read_database_uri(uri=db_uri, query=messages_query, engine="adbc")

In [ ]:
result_dict = conversations[["id", "interviewer"]].to_dict(as_series=False)
id_interviewer_dict = dict(zip(result_dict["id"], result_dict["interviewer"]))

In [ ]:
messages = messages.filter(pl.col("conversation_id").is_in(conversations["id"]))

In [ ]:
messages = (
    messages.with_columns(
        created_at=pl.col("created_at").str.to_datetime(
            format="%Y-%m-%d %H:%M:%S%.6f", time_unit="ms"
        )
    )
    # Create a time delta column, which indicates the time since first messsage
    # for each conversation
    .lazy()
    .sort(["created_at"])
    .with_columns(
        pl.col("created_at").min().over("conversation_id").alias("first_created_at")
    )
    .with_columns(
        (pl.col("created_at") - pl.col("first_created_at")).alias("time_delta")
    )
    .drop("first_created_at")
    .collect()
).sort(["conversation_id", "message_id"])

In [ ]:
conversation_ids = messages["conversation_id"].unique()
last_conversation_id = messages["conversation_id"].max()

In [ ]:
conversation_iter = iter(conversation_ids)

In [ ]:
try:
    conversation_id = next(conversation_iter)
    print_interview(
        conversation_id=conversation_id,
        messages=messages,
        timestamp_format="%d-%m %H:%M:%S",
        interviewer="ai",
    )
except StopIteration:
    print("No more conversations to display")

## General patterns

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure()

conversation_ids = messages["conversation_id"].unique()
for conversation_id in conversation_ids:
    df = (
        messages.filter(conversation_id=conversation_id, role="ASSISTANT")
        .with_columns(
            total_seconds=pl.col("time_delta").dt.total_seconds(),
        )
        .with_columns(
            total_minutes=pl.col("total_seconds") / 60,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df["message_id"],
            y=df["total_minutes"],
            mode="lines",
            name=f"Conversation {conversation_id}",
        )
    )

fig.update_layout(
    title="Time since beginning of conversation for each message",
    xaxis_title="Message ID",
    yaxis_title="Total Minutes",
    yaxis=dict(range=[0, None], side="right", tickprefix="  "),
    showlegend=False,
)

fig.update_yaxes()
fig.show()
html = fig.to_html("time_since_beginning.html", full_html=False, include_plotlyjs=None)

In [ ]:
# Remove introduction and consent messages.
messages = messages.filter(messages["message_id"] > 4)

In [ ]:
assistant_content = messages.filter(role="ASSISTANT")["content"]
assistant_content_length_mean = assistant_content.str.len_chars().mean()
assistant_content_length_std = assistant_content.str.len_chars().std()

user_content = messages.filter(role="USER")["content"]
user_content_length_mean = user_content.str.len_chars().mean()
user_content_length_std = user_content.str.len_chars().std()

assistant_response_time = messages

print(
    f"Number of interviews: {messages['conversation_id'].n_unique()}\n"
    f"Average question length: {assistant_content_length_mean:.2f} ({assistant_content_length_std:.2f})\n"
    f"Average answer length: {user_content_length_mean:.2f} ({user_content_length_std:.2f})\n"
)

In [ ]:
def get_average_response_times(
    who: Literal["interviewer"] | Literal["user"], stat=np.mean
) -> list[float]:
    average_response_times = []
    for conversation_id in conversation_ids:
        conv_df = messages.filter(conversation_id=conversation_id)

        response_times = calculate_response_times(conv_df, who=who)

        average_response_times.append(stat(response_times).item().total_seconds())

    return average_response_times

In [ ]:
average_response_times = get_average_response_times("user", stat=np.median)

df = pl.DataFrame({"average_response_times": average_response_times})

fig = px.histogram(
    df,
    x="average_response_times",
    nbins=100,
    title="Median interviewer response time",
    labels={"average_response_times": "seconds"},
)

fig.update_layout(
    xaxis_title="Seconds",
    yaxis_title="Count",
)

fig.show()

In [ ]:
# Assuming messages is a DataFrame
df = (
    messages.group_by("conversation_id")
    .agg(pl.count("conversation_id").alias("len"))
    .to_pandas()
)

fig = px.histogram(
    df,
    x="len",
    title="Interview length",
    labels={"len": "Number of messages"},
)

fig.update_layout(
    xaxis_title="Number of messages",
    yaxis_title="Count",
)

fig.show()

In [ ]:
df = (
    messages.filter(role="ASSISTANT")
    .select(pl.col("content").str.len_chars())
    .to_pandas()
)

fig = px.histogram(
    df,
    x="content",
    nbins=20,
    title="Interviewer",
    labels={"content": "Message lengths"},
)

fig.update_layout(
    xaxis_title="Message lengths",
    yaxis_title="Counts",
)

fig.show()

In [ ]:
# Assuming messages is a DataFrame
df = messages.filter(role="USER").select(pl.col("content").str.len_chars()).to_pandas()

fig = px.histogram(
    df,
    x="content",
    nbins=20,
    title="User",
    labels={"content": "Message lengths"},
)

fig.update_layout(
    xaxis_title="Message lengths",
    yaxis_title="Counts",
)

fig.show()